In [7]:
"""
Figures comparing dynamic (feedforward) vs post-selected (static) protocols.
Run after dynamic_circuit_pipeline.py has produced dynamic_circuit_results.json.

Figures:
  fig_dyn1_fidelity_comparison.pdf  — F side-by-side with bootstrap CIs
  fig_dyn2_bloch_comparison.pdf     — Bloch sphere overlay (ideal / ps / dynamic)
  fig_dyn3_shotcost_tradeoff.pdf    — Shot cost vs useful samples tradeoff curves
  fig_dyn4_bootstrap_overlay.pdf    — Bootstrap F distributions overlaid
  fig_dyn5_densitymatrix.pdf        — Re/Im ρ_Y for both protocols vs ideal
  fig_dyn_panel.pdf                 — Combined panel for paper
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

# ── Load data ─────────────────────────────────────────────────────────────────
with open('/Users/nandan/Desktop/CTCs/IBM/dynamic_circuit_results.json') as f:
    D = json.load(f)

ps  = D['postselected']
dyn = D['dynamic']
trd = D['tradeoff']
shots = D['metadata']['shots']

# ── Ideal reference ───────────────────────────────────────────────────────────
sx_ideal=-0.2384; sy_ideal=0.5179; sz_ideal=-0.8215
rho_M_re=[[0.08923,-0.11921],[-0.11921, 0.91077]]
rho_M_im=[[0.0,   -0.25895],[ 0.25895, 0.0    ]]

def load_rho(d):
    return np.array([[d['rho_Y'][i][j]['re']+1j*d['rho_Y'][i][j]['im']
                      for j in range(2)] for i in range(2)])

rho_ps  = load_rho(ps)
rho_dyn = load_rho(dyn)

bsF_ps  = np.array(ps['bootstrap_F'])
bsF_dyn = np.array(dyn['bootstrap_F'])

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 9.5, 'figure.dpi': 180,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8, 'pdf.fonttype': 42,
})

C_PS    = '#C0392B'   # deep red   — post-selected
C_DYN   = '#2980B9'   # blue       — dynamic
C_IDEAL = '#27AE60'   # green      — ideal
OUT     = '/Users/nandan/Desktop/CTCs/IBM/Figures_PCTC_Hard'

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 1 — Fidelity comparison bar chart
# ═══════════════════════════════════════════════════════════════════════════════
def fig_fidelity():
    fig, ax = plt.subplots(figsize=(5.0, 3.8))

    protocols = ['Ideal\n(theory)', 'Post-selected\n(static)', 'Dynamic\n(feedforward)']
    F_vals    = [1.0, ps['fidelity'], dyn['fidelity']]
    colors    = [C_IDEAL, C_PS, C_DYN]
    elo = [0, ps['fidelity']-ps['fidelity_ci'][0], dyn['fidelity']-dyn['fidelity_ci'][0]]
    ehi = [0, ps['fidelity_ci'][1]-ps['fidelity'], dyn['fidelity_ci'][1]-dyn['fidelity']]
    hatches = ['///', '', '']

    x    = np.arange(3)
    bars = ax.bar(x, F_vals, color=colors, width=0.52,
              zorder=3, edgecolor='white')
    for bar, h in zip(bars, hatches):
        bar.set_hatch(h)

    ax.axhline(1.0, color=C_IDEAL, ls='--', lw=1.1, alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(protocols, fontsize=10)
    ax.set_ylim(0.70, 1.08)
    ax.set_ylabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title('Output fidelity comparison\n'
                 '(ibm_torino · 95% bootstrap CI)', pad=8)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)

    for xi, fv, el, eh in zip(x[1:], F_vals[1:], elo[1:], ehi[1:]):
        offset = 0.009 if xi == 2 else 0.004
        ax.text(xi, fv + offset, f'{fv:.4f}', ha='center', fontsize=9.5, color='#111')
        # ax.text(xi, fv - 0.012, f'[{fv-el:.4f}, {fv+eh:.4f}]',ha='center', fontsize=7.5, color='#555')

    delta = dyn['fidelity'] - ps['fidelity']
    sign  = '+' if delta >= 0 else ''
    ax.annotate('', xy=(2, dyn['fidelity']), xytext=(1, ps['fidelity']),
                arrowprops=dict(arrowstyle='<->', color='black', lw=1.5,
                connectionstyle='arc3,rad=0.2'))
    ax.text(1.5, (dyn['fidelity']+ps['fidelity'])/2 + 0.012,
            f'$\\Delta F={sign}{delta:.4f}$',
            ha='center', fontsize=9, color='#444',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#ccc', lw=0.7))

    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 2 — Bloch sphere: ideal / post-selected / dynamic
# ═══════════════════════════════════════════════════════════════════════════════
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0),(0,0),*args,**kwargs)
        self._verts3d = xs,ys,zs
    def do_3d_projection(self, renderer=None):
        xs,ys,zs=proj3d.proj_transform(*self._verts3d,self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        return np.min(zs)

def draw_bloch(ax, vectors, labels, colors, linestyles=None):
    if linestyles is None: linestyles=['solid']*len(vectors)
    u=np.linspace(0,2*np.pi,60); v=np.linspace(0,np.pi,40)
    ax.plot_wireframe(np.outer(np.cos(u),np.sin(v)),
                      np.outer(np.sin(u),np.sin(v)),
                      np.outer(np.ones(60),np.cos(v)),
                      rstride=5,cstride=5,color='#cccccc',lw=0.3,alpha=0.35)
    for xyz,lbl in [([1.25,0,0],'$x$'),([0,1.25,0],'$y$'),([0,0,1.25],'$z$'),
                    ([-1.25,0,0],''),([ 0,-1.25,0],''),([ 0,0,-1.25],'')]:
        ax.plot([0,xyz[0]],[0,xyz[1]],[0,xyz[2]],color='#999',lw=0.7,alpha=0.7)
        if lbl: ax.text(xyz[0]*1.12,xyz[1]*1.12,xyz[2]*1.12,lbl,fontsize=10,ha='center',color='#555')
    t=np.linspace(0,2*np.pi,120)
    for c1,c2,c3 in [(np.cos(t),np.sin(t),np.zeros(120)),
                     (np.cos(t),np.zeros(120),np.sin(t)),
                     (np.zeros(120),np.cos(t),np.sin(t))]:
        ax.plot(c1,c2,c3,color='#bbb',lw=0.5,alpha=0.4)
    for (bx,by,bz),lbl,col in zip(vectors,labels,colors):
        arw=Arrow3D([0,bx],[0,by],[0,bz],mutation_scale=15,lw=2.3,
                    arrowstyle='-|>',color=col)
        ax.add_artist(arw)
        ax.text(bx*1.15,by*1.15,bz*1.15,lbl,fontsize=9.5,color=col,fontweight='bold')
    ax.set_xlim(-1.35,1.35); ax.set_ylim(-1.35,1.35); ax.set_zlim(-1.35,1.35)
    ax.set_box_aspect([1,1,1]); ax.axis('off')

def fig_bloch():
    fig = plt.figure(figsize=(5.2, 4.8))
    ax  = fig.add_subplot(111, projection='3d')
    draw_bloch(ax,
        [[sx_ideal,sy_ideal,sz_ideal],
         [ps['bloch']['sx'],  ps['bloch']['sy'],  ps['bloch']['sz']],
         [dyn['bloch']['sx'], dyn['bloch']['sy'], dyn['bloch']['sz']]],
        ['$\\rho_M$', '$\\rho_Y^{\\rm ps}$', '$\\rho_Y^{\\rm dyn}$'],
        [C_IDEAL, C_PS, C_DYN])
    patches = [
        mpatches.Patch(color=C_IDEAL, label='Ideal $\\rho_M$'),
        mpatches.Patch(color=C_PS,    label='Post-selected $\\rho_Y$'),
        mpatches.Patch(color=C_DYN,   label='Dynamic $\\rho_Y$'),
    ]
    ax.legend(handles=patches, loc='upper left',
              bbox_to_anchor=(-0.1, 1.0), fontsize=9, framealpha=0.9)
    r_ps  = (ps['bloch']['sx']**2  + ps['bloch']['sy']**2  + ps['bloch']['sz']**2)**0.5
    r_dyn = (dyn['bloch']['sx']**2 + dyn['bloch']['sy']**2 + dyn['bloch']['sz']**2)**0.5
    ax.text2D(0.02, 0.05,
              f'$|\\mathbf{{r}}|_{{\\rm ideal}}=1.000$\n'
              f'$|\\mathbf{{r}}|_{{\\rm ps}}={r_ps:.3f}$\n'
              f'$|\\mathbf{{r}}|_{{\\rm dyn}}={r_dyn:.3f}$',
              transform=ax.transAxes, fontsize=8.5,
              bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ccc',lw=0.6))
    ax.set_title('(b) Bloch sphere comparison', fontsize=12, pad=4)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 3 — Shot-cost tradeoff curves
# ═══════════════════════════════════════════════════════════════════════════════
def fig_shotcost():
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))

    N_useful = np.logspace(1, 4, 300)

    # Post-selected: total shots = N_useful / p_succ
    shots_ps  = N_useful / ps['p_succ']
    # Dynamic: total shots = N_useful (all kept)
    shots_dyn = N_useful

    # ── Left: total shots required ────────────────────────────────────────────
    ax = axes[0]
    ax.loglog(N_useful, shots_ps,  color=C_PS,  lw=2.2, label='Post-selected')
    ax.loglog(N_useful, shots_dyn, color=C_DYN, lw=2.2, label='Dynamic (feedforward)')
    ax.fill_between(N_useful, shots_dyn, shots_ps, alpha=0.12, color=C_DYN,
                    label=f'Shot savings  ({trd["shot_cost_ratio"]:.2f}×)')
    ax.axvline(ps['n_kept'], color=C_PS, ls=':', lw=1.2, alpha=0.7,
               label=f'Actual $N_{{\\rm kept}}={ps["n_kept"]:,}$')
    ax.set_xlabel('Target useful output samples $N_{\\rm useful}$')
    ax.set_ylabel('Total shots required')
    ax.set_title('Shot cost to achieve $N_{\\rm useful}$ samples', pad=8)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.xaxis.grid(True, alpha=0.3)
    # Annotate cost ratio at N=1000
    n_ref = 1000
    y_ps  = n_ref / ps['p_succ']
    y_dyn = n_ref
    ax.annotate(f'{trd["shot_cost_ratio"]:.2f}×',
                xy=(n_ref, (y_ps+y_dyn)/2),
                fontsize=10, color=C_DYN, ha='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=C_DYN, lw=0.8))

    # ── Right: fidelity vs shots for both (using bootstrap std as proxy) ──────
    ax2 = axes[1]
    # Model: F(N) ≈ F_asymptotic - noise_floor * exp(-N/tau)
    # Use measured F and std to anchor — this is illustrative
    N_arr   = np.linspace(100, 20000, 400)
    F_ps_curve  = ps['fidelity']  - (1-ps['fidelity'])   * np.exp(-N_arr / (ps['n_kept']/2))
    F_dyn_curve = dyn['fidelity'] - (1-dyn['fidelity']) * np.exp(-N_arr / (dyn['n_kept']/2))
    F_ps_curve  = np.clip(F_ps_curve,  0, 1)
    F_dyn_curve = np.clip(F_dyn_curve, 0, 1)

    ax2.plot(N_arr, F_ps_curve,  color=C_PS,    lw=2.0, label='Post-selected (model)')
    ax2.plot(N_arr, F_dyn_curve, color=C_DYN,   lw=2.0, label='Dynamic (model)')
    ax2.axhline(1.0, color=C_IDEAL, ls=':', lw=1.3, alpha=0.7, label='Ideal $F=1$')

    # Plot measured points with CI bars
    ax2.errorbar(ps['n_kept'],  ps['fidelity'],
                 yerr=[[ps['fidelity']-ps['fidelity_ci'][0]],
                       [ps['fidelity_ci'][1]-ps['fidelity']]],
                 fmt='o', color=C_PS,  ms=7, capsize=5, zorder=5,
                 label=f'Measured ps  $F={ps["fidelity"]:.4f}$')
    ax2.errorbar(dyn['n_kept'], dyn['fidelity'],
                 yerr=[[dyn['fidelity']-dyn['fidelity_ci'][0]],
                       [dyn['fidelity_ci'][1]-dyn['fidelity']]],
                 fmt='s', color=C_DYN, ms=7, capsize=5, zorder=5,
                 label=f'Measured dyn $F={dyn["fidelity"]:.4f}$')

    ax2.set_xlabel('Number of useful (kept) shots $N_{\\rm kept}$')
    ax2.set_ylabel('$F(\\rho_Y,\\, \\rho_M)$')
    ax2.set_title('Fidelity–shot-cost tradeoff', pad=8)
    ax2.legend(fontsize=8.5, framealpha=0.9)
    ax2.yaxis.grid(True, alpha=0.3)
    ax2.set_ylim(0.5, 1.05)

    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 4 — Bootstrap distyeah ributions overlaid
# ═══════════════════════════════════════════════════════════════════════════════
def fig_bootstrap():
    fig, ax = plt.subplots(figsize=(5.5, 3.8))

    kw = dict(bins=50, alpha=0.65, edgecolor='white', linewidth=0.4, zorder=3)
    ax.hist(bsF_ps,  color=C_PS,  label='Post-selected', **kw)
    ax.hist(bsF_dyn, color=C_DYN, label='Dynamic', **kw)

    for bsF, col, F, ci in [
        (bsF_ps,  C_PS,  ps['fidelity'],  ps['fidelity_ci']),
        (bsF_dyn, C_DYN, dyn['fidelity'], dyn['fidelity_ci'])]:
        med = np.median(bsF)
        ax.axvline(med,   color=col, lw=2.0, zorder=4)
        ax.axvline(ci[0], color=col, lw=1.1, ls='--', zorder=4)
        ax.axvline(ci[1], color=col, lw=1.1, ls='--', zorder=4)
        ax.axvspan(ci[0], ci[1], alpha=0.10, color=col, zorder=1)

    ax.axvline(1.0, color=C_IDEAL, lw=1.5, ls=':', zorder=5, label='Ideal $F=1$')
    ax.set_xlabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_ylabel('Bootstrap count')
    ax.set_title('(e) Bootstrap $F$ distributions\n'
                 '(vertical lines = median and 95% CI boundaries)', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc')
    ax.yaxis.grid(True, alpha=0.3, zorder=0)

    # Annotation boxes
    for bsF, col, label, xoff in [
            (bsF_ps,  C_PS,  'ps',  -0.04),
            (bsF_dyn, C_DYN, 'dyn', +0.01)]:
        med=np.median(bsF); lo=np.percentile(bsF,2.5); hi=np.percentile(bsF,97.5)
        ymax=ax.get_ylim()[1]

    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 5 — Density matrix comparison
# ═══════════════════════════════════════════════════════════════════════════════
def fig_densitymatrix():
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))
    elems = [(0,0),(0,1),(1,0),(1,1)]
    xlbls = ['$|00\\rangle$','$|01\\rangle$','$|10\\rangle$','$|11\\rangle$']
    x = np.arange(4)

    for ax, part, title, ideal_arr in zip(
            axes,
            ['re','im'],
            ['(f) $\\mathrm{Re}(\\rho_Y)$', '(g) $\\mathrm{Im}(\\rho_Y)$'],
            [rho_M_re, rho_M_im]):

        ideal = [ideal_arr[i][j] for i,j in elems]
        ps_v  = [getattr(rho_ps[i,j], 'real' if part=='re' else 'imag')
                 for i,j in elems]
        dyn_v = [getattr(rho_dyn[i,j], 'real' if part=='re' else 'imag')
                 for i,j in elems]

        w = 0.22; nb = 3
        off = np.array([-(nb-1)/2 + i for i in range(nb)]) * w

        ax.bar(x+off[0], ideal, width=w, color=C_IDEAL,
               label='Ideal ($\\rho_M$)', edgecolor='white', zorder=3, hatch='///')
        ax.bar(x+off[1], ps_v,  width=w, color=C_PS,
               label='Post-selected',    edgecolor='white', zorder=3)
        ax.bar(x+off[2], dyn_v, width=w, color=C_DYN,
               label='Dynamic',          edgecolor='white', zorder=3)

        ax.axhline(0, color='#555', lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, fontsize=9.5)
        ax.set_title(title, fontsize=12)
        ax.set_ylabel('Matrix element value')
        ax.yaxis.grid(True, alpha=0.3, zorder=0)
        ax.set_ylim(-0.75, 1.10)
        ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9)

    fig.suptitle('Density matrix comparison: ideal vs post-selected vs dynamic\n'
                 '(ibm\\_torino, post-selected tomography)', fontsize=11, y=1.02)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  COMBINED PANEL
# ═══════════════════════════════════════════════════════════════════════════════
def fig_panel():
    fig = plt.figure(figsize=(15.0, 9.5))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.46, wspace=0.36)

    # (a) Fidelity bars
    ax1 = fig.add_subplot(gs[0,0])
    protos=['Ideal','Post-sel.','Dynamic']
    Fv=[1.0,ps['fidelity'],dyn['fidelity']]; cols=[C_IDEAL,C_PS,C_DYN]
    elo=[0,ps['fidelity']-ps['fidelity_ci'][0],dyn['fidelity']-dyn['fidelity_ci'][0]]
    ehi=[0,ps['fidelity_ci'][1]-ps['fidelity'],dyn['fidelity_ci'][1]-dyn['fidelity']]
    bars=ax1.bar(range(3),Fv,color=cols,width=0.5,
                 yerr=[elo,ehi],error_kw=dict(elinewidth=1.4,capsize=5,capthick=1.4,ecolor='#222'),
                 zorder=3,edgecolor='white')
    bars[0].set_hatch('///')
    ax1.axhline(1.0,color=C_IDEAL,ls='--',lw=1.1,alpha=0.6)
    ax1.set_xticks(range(3)); ax1.set_xticklabels(protos,fontsize=10)
    ax1.set_ylim(0.70,1.08); ax1.set_ylabel('$F(\\rho_Y,\\rho_M)$')
    ax1.set_title('(a) Fidelity comparison',fontsize=11)
    ax1.yaxis.grid(True,alpha=0.3,zorder=0)
    for xi,fv in zip(range(3),Fv): ax1.text(xi,fv+0.004,f'{fv:.4f}',ha='center',fontsize=9)
    delta=dyn['fidelity']-ps['fidelity']; sign='+' if delta>=0 else ''
    ax1.text(1.5,0.74,f'$\\Delta F={sign}{delta:.4f}$',ha='center',fontsize=9.5,
             color='#333',bbox=dict(boxstyle='round,pad=0.25',fc='white',ec='#ccc',lw=0.7))

    # (b) Shot-cost tradeoff
    ax2 = fig.add_subplot(gs[0,1])
    Nu=np.logspace(1,4.5,300)
    ax2.loglog(Nu,Nu/ps['p_succ'],color=C_PS, lw=2.0,label='Post-selected')
    ax2.loglog(Nu,Nu,             color=C_DYN,lw=2.0,label='Dynamic')
    ax2.fill_between(Nu,Nu,Nu/ps['p_succ'],alpha=0.1,color=C_DYN,
                     label=f'{trd["shot_cost_ratio"]:.2f}× savings')
    ax2.set_xlabel('Target $N_{\\rm useful}$'); ax2.set_ylabel('Total shots')
    ax2.set_title('(b) Shot-cost tradeoff',fontsize=11)
    ax2.legend(fontsize=9,framealpha=0.9); ax2.grid(True,alpha=0.25)

    # (c) Bootstrap overlay
    ax3 = fig.add_subplot(gs[0,2])
    kw=dict(bins=50,alpha=0.65,edgecolor='white',lw=0.4,zorder=3)
    ax3.hist(bsF_ps,color=C_PS,label='Post-selected',**kw)
    ax3.hist(bsF_dyn,color=C_DYN,label='Dynamic',**kw)
    ax3.axvline(1.0,color=C_IDEAL,lw=1.5,ls=':',zorder=5,label='Ideal')
    for bsF_,col_ in [(bsF_ps,C_PS),(bsF_dyn,C_DYN)]:
        ax3.axvline(np.median(bsF_),color=col_,lw=1.8,zorder=4)
        lo_,hi_=np.percentile(bsF_,2.5),np.percentile(bsF_,97.5)
        ax3.axvline(lo_,color=col_,lw=1.0,ls='--',zorder=4)
        ax3.axvline(hi_,color=col_,lw=1.0,ls='--',zorder=4)
    ax3.set_xlabel('$F(\\rho_Y,\\rho_M)$'); ax3.set_ylabel('Count')
    ax3.set_title('(c) Bootstrap distributions',fontsize=11)
    ax3.legend(fontsize=9,framealpha=0.9); ax3.yaxis.grid(True,alpha=0.3,zorder=0)

    # (d) Bloch sphere
    ax4 = fig.add_subplot(gs[1,0],projection='3d')
    draw_bloch(ax4,
               [[sx_ideal,sy_ideal,sz_ideal],
                [ps['bloch']['sx'],ps['bloch']['sy'],ps['bloch']['sz']],
                [dyn['bloch']['sx'],dyn['bloch']['sy'],dyn['bloch']['sz']]],
               ['$\\rho_M$','$\\rho_Y^{\\rm ps}$','$\\rho_Y^{\\rm dyn}$'],
               [C_IDEAL,C_PS,C_DYN])
    p_=[mpatches.Patch(color=c,label=l) for c,l in
        zip([C_IDEAL,C_PS,C_DYN],['Ideal','Post-sel.','Dynamic'])]
    ax4.legend(handles=p_,loc='upper left',bbox_to_anchor=(-0.12,1.0),fontsize=8.5,framealpha=0.9)
    ax4.set_title('(d) Bloch sphere',fontsize=11,pad=2)

    # (e)/(f) Density matrix Re/Im
    for col_idx,(part,title,ideal_arr) in enumerate([
            ('re','(e) $\\mathrm{Re}(\\rho_Y)$',rho_M_re),
            ('im','(f) $\\mathrm{Im}(\\rho_Y)$',rho_M_im)]):
        ax=fig.add_subplot(gs[1,1+col_idx])
        elems=[(0,0),(0,1),(1,0),(1,1)]
        xlbls_=['$|00\\rangle$','$|01\\rangle$','$|10\\rangle$','$|11\\rangle$']
        iv=[ideal_arr[i][j] for i,j in elems]
        pv=[getattr(rho_ps[i,j],'real' if part=='re' else 'imag') for i,j in elems]
        dv=[getattr(rho_dyn[i,j],'real' if part=='re' else 'imag') for i,j in elems]
        xd=np.arange(4); wd=0.22
        off_=np.array([-1,0,1])*wd
        ax.bar(xd+off_[0],iv,width=wd,color=C_IDEAL,edgecolor='white',zorder=3,hatch='///',label='Ideal')
        ax.bar(xd+off_[1],pv,width=wd,color=C_PS,   edgecolor='white',zorder=3,label='Post-sel.')
        ax.bar(xd+off_[2],dv,width=wd,color=C_DYN,  edgecolor='white',zorder=3,label='Dynamic')
        ax.axhline(0,color='#555',lw=0.8)
        ax.set_xticks(xd); ax.set_xticklabels(xlbls_,fontsize=8.5)
        ax.set_title(title,fontsize=11); ax.set_ylabel('Matrix element')
        ax.yaxis.grid(True,alpha=0.3,zorder=0); ax.set_ylim(-0.75,1.10)
        ax.legend(fontsize=8,framealpha=0.9)

    fig.suptitle(
        f'Dynamic circuit vs post-selected comparison — ibm\\_torino  '
        f'($N_{{\\rm shots}}={shots:,}$/basis)\n'
        f'Post-selected: $F={ps["fidelity"]:.4f}$, '
        f'Dynamic: $F={dyn["fidelity"]:.4f}$, '
        f'Shot-cost ratio: {trd["shot_cost_ratio"]:.2f}×',
        fontsize=11.5, y=1.005
    )
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  Save all figures
# ═══════════════════════════════════════════════════════════════════════════════
print('Generating dynamic circuit comparison figures ...')
figs = [
    ('fig_dyn1_fidelity_comparison.pdf', fig_fidelity),
    ('fig_dyn2_bloch_comparison.pdf',    fig_bloch),
    ('fig_dyn3_shotcost_tradeoff.pdf',   fig_shotcost),
    ('fig_dyn4_bootstrap_overlay.pdf',   fig_bootstrap),
    ('fig_dyn5_densitymatrix.pdf',       fig_densitymatrix),
    ('fig_dyn_panel.pdf',                fig_panel),
]
for fname, fn in figs:
    f = fn()
    f.savefig(OUT + fname, bbox_inches='tight', dpi=200)
    plt.close(f)
    print(f'  [✓] {fname}')

print(f'\nKey results:')
print(f'  Post-selected: F={ps["fidelity"]:.4f}  [{ps["fidelity_ci"][0]:.4f},{ps["fidelity_ci"][1]:.4f}]')
print(f'  Dynamic:       F={dyn["fidelity"]:.4f}  [{dyn["fidelity_ci"][0]:.4f},{dyn["fidelity_ci"][1]:.4f}]')
print(f'  ΔF:            {dyn["fidelity"]-ps["fidelity"]:+.4f}')
print(f'  Shot savings:  {trd["shot_cost_ratio"]:.2f}×')

Generating dynamic circuit comparison figures ...
  [✓] fig_dyn1_fidelity_comparison.pdf
  [✓] fig_dyn2_bloch_comparison.pdf
  [✓] fig_dyn3_shotcost_tradeoff.pdf
  [✓] fig_dyn4_bootstrap_overlay.pdf
  [✓] fig_dyn5_densitymatrix.pdf
  [✓] fig_dyn_panel.pdf

Key results:
  Post-selected: F=0.8365  [0.8195,0.8537]
  Dynamic:       F=0.7465  [0.7379,0.7551]
  ΔF:            -0.0899
  Shot savings:  4.19×
